In [ ]:
import os

if os.path.exists("NovaS.pdf"):
    print("Successfully found the file.")
else:   
    print("File not found.")

File not found.


In [17]:
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_experimental.text_splitter import SemanticChunker
#from langchain_text_splitters import RecursiveCharacterTextSplitter

import os
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import OpenAIEmbeddings

##### STEP-0: LOAD THE PDF INTO THE TEXT FORMAT

In [18]:
docs = PyPDFLoader("../CH-2_RAG/NovaS.pdf").load()

full_text = "\n".join([p.page_content for p in docs])
full_text

'NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future. At the \nbeginning, the company did not have large investments or a big office. Instead, it started \nwith only six employees working together in a small shared workspace. The founders were \nnot focused on becoming successful overnight. Their main goal was to build strong \ntechnical knowledge, gain practical experience, and slowly grow by delivering real value to \ntheir clients. Most of the early work involved helping small companies understand their \nexisting data and use simple reporting solutions to make better business decisions. \nDuring the first year of operations, the company worked mostly with local startups that did \nnot have large 

##### STEP-1: CREATING CHUNKS OF THE TEXT DATA

In [19]:
chunker = SemanticChunker(
    OpenAIEmbeddings(model = "text-embedding-3-small"),
    breakpoint_threshold_type="percentile", # To calculate the percenatge of similarity based on the previous chunks
    breakpoint_threshold_amount=60
)

semantic_chunks = chunker.create_documents([full_text])
semantic_chunks

[Document(metadata={}, page_content='NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future.'),
 Document(metadata={}, page_content='At the \nbeginning, the company did not have large investments or a big office.'),
 Document(metadata={}, page_content='Instead, it started \nwith only six employees working together in a small shared workspace. The founders were \nnot focused on becoming successful overnight.'),
 Document(metadata={}, page_content='Their main goal was to build strong \ntechnical knowledge, gain practical experience, and slowly grow by delivering real value to \ntheir clients. Most of the early work involved helping small companies understand their \nexisting data and use simple reporting 

##### STEP-2&3: CREATING EMBEDDINGS OF THE CHUNKS & STORING THEM IN VECTOR DB 

In [20]:
from langchain_community.vectorstores import Chroma


In [21]:
import chromadb
import chromadb.config
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
embed_model = OpenAIEmbeddings(model="text-embedding-3-small")

chroma_db = Chroma.from_documents(semantic_chunks, embed_model, persist_directory="./chroma_db_semantic")




#### STEP-4: CONNECTION & RETRIEVAL OF CHROMA DB

In [22]:
chroma_db_con = Chroma(persist_directory="./chroma_db_semantic", embedding_function=embed_model)


In [23]:
chroma_db_con.similarity_search("By 2019, How many employees were there?", k=3)

[Document(metadata={}, page_content='The \nnumber of employees increased again, and the company also started hiring people who \nhad experience in big data technologies. This helped the organization build stronger \ntechnical capabilities and handle larger clients. Another important milestone came in 2022 when the organization decided to invest more \ntime in research and learning new technologies. Instead of working only on client projects, \nthe company created a small internal research group.'),
 Document(metadata={}, page_content='The organization began working with \ncompanies from different industries such as retail, education, and small healthcare \nproviders. Each industry had different challenges, and this helped the team improve its \nunderstanding of real-world business problems. The company also started focusing more \non improving the quality of its work instead of simply increasing the number of projects. The founders believed that long-term growth would come only if the 

#### STEP-5: LLM INTEGRATION AND ANSWER GENERATION

In [26]:
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

In [ ]:
user_query = "By 2020, how many employees were there?"
rel_chunks = chroma_db_con.similarity_search(user_query, k=3)

rel_chunks_content = []
for i, chunk in enumerate(rel_chunks):
    rel_chunks_content.append(chunk.page_content)
    
llm.invoke(f"{user_query}, Use the following context to answer the question: {str(rel_chunks_content)}")
           
    

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/Users/abhishekkumar/anaconda3/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3526, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/var/folders/c4/y4y568g51bsbt1rt9m950y3c0000gn/T/ipykernel_67772/2134132942.py", line 7, in <module>
    llm.invoke(f"{user_query}, Use the following context to answer the question: {str(rel_chunks_content)}")
  File "/Users/abhishekkumar/anaconda3/lib/python3.11/site-packages/langchain_core/language_models/chat_models.py", line 454, in invoke
    else:
          
  File "/Users/abhishekkumar/anaconda3/lib/python3.11/site-packages/langchain_core/language_models/chat_models.py", line 1185, in generate_prompt
    **kwargs: Arbitrary additional keyword arguments. These are usually passed
       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/abhishekkumar/anaconda3/lib/python3.11/site-packages/langchain_core/language_models/chat_m